# Лекция 8. Оптимизация моделей для инференса — код

Код из лекции: правильные замеры времени и калибровки, дистилляция знаний, квантизация (ручная, калибровка активаций, QAT, INT8 и BF16 в PyTorch), прунинг (нерегулярный, структурный, 2:4), низкоранговое разложение, свёртка BatchNorm, TorchScript, экспорт в ONNX и ONNX Runtime, Test-Time Augmentation и MC-Dropout.

Сквозной пример — «большая» модель-**учитель** для цифр 8×8: два скрытых слоя по 512 нейронов, около 300 тысяч параметров. Для цифр она избыточна, и на ней видно, сколько можно убрать без потери качества.

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все». Видеокарта не нужна: весь ноутбук выполняется на процессоре примерно за полминуты.

**Время в миллисекундах у вас будет другим** — оно зависит от процессора Colab, и соотношения тоже могут измениться: например, INT8 на процессоре Intel или AMD может оказаться быстрее FP32, хотя в лекции он медленнее. Это один из уроков лекции: ускорение меряют на своём железе. Точность и последние знаки других чисел тоже могут немного отличаться.

Первая ячейка ставит `onnx` и `onnxruntime` — они нужны в главе 5. На своём компьютере понадобятся `pip install torch scikit-learn onnx onnxruntime onnxscript`.

In [ ]:
!pip install -q onnx onnxruntime onnxscript

## Глава 1. Что и как измерять

### 1.3. Учитель и первые измерения

Данные, учитель и инструменты измерения. `mlp(width, depth)` строит сеть, `train` — цикл обучения (параметр `loss_fn` позволит подменить потерю), `size_kb` — размер сохранённых весов, `latency` — 20 вызовов прогрева, затем 300 замеров; возвращает медиану, p95 и p99 в миллисекундах.

In [ ]:
import copy
import io
import os
import time
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(4)                              # одинаковые условия на разных машинах

# 1797 картинок 8×8, каждая вытянута в строку из 64 пикселей
X_all, y_all = load_digits(return_X_y=True)
X_all = torch.tensor(X_all, dtype=torch.float32) / 16   # яркость от 0 до 1
y_all = torch.tensor(y_all)
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=0, stratify=y_all)

def mlp(width, depth=2, p_drop=0.2):
    """Сеть: 64 пикселя → depth скрытых слоёв ширины width (ReLU и dropout) → 10 логитов."""
    layers, d = [], 64
    for _ in range(depth):
        layers += [nn.Linear(d, width), nn.ReLU(), nn.Dropout(p_drop)]
        d = width
    return nn.Sequential(*layers, nn.Linear(d, 10))

def train(model, X=X_train, y=y_train, epochs=100, lr=1e-3, loss_fn=None, seed=0):
    """Цикл обучения; loss_fn(model, xb, b) подменяет потерю (b — номера примеров батча в X)."""
    torch.manual_seed(seed)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    model.train()
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            b = order[i:i + 64]
            loss = loss_fn(model, X[b], b) if loss_fn else F.cross_entropy(model(X[b]), y[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
    model.eval()                                      # после обучения — режим инференса
    return model

def accuracy(model, X=X_test, y=y_test):
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def size_kb(model):
    """Размер сохранённых весов в килобайтах — столько модель займёт на диске."""
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1024

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def latency(fn, x, warmup=20, runs=300):
    """Время одного вызова fn(x) в миллисекундах: медиана, p95 и p99 после прогрева."""
    with torch.no_grad():
        for _ in range(warmup):                       # прогрев: первые вызовы медленнее
            fn(x)
        times = []
        for _ in range(runs):
            t = time.perf_counter()
            fn(x)
            times.append((time.perf_counter() - t) * 1000)
    times = torch.tensor(times)
    return times.median().item(), times.quantile(0.95).item(), times.quantile(0.99).item()

torch.manual_seed(0)
teacher = train(mlp(512))                             # «большая» модель: два слоя по 512 нейронов
x1 = X_test[:1]                                       # один запрос — одна картинка
med, p95, p99 = latency(teacher, x1)
print(f"учитель: {n_params(teacher):,} параметров | {size_kb(teacher):.0f} КБ | точность {accuracy(teacher):.3f}")
print(f"задержка на один запрос: медиана {med:.3f} мс, p95 {p95:.3f} мс, p99 {p99:.3f} мс")

**Что получилось.** Учитель весит около 1.2 МБ (300 тысяч чисел по 4 байта), точность 0.980, один запрос — доли миллисекунды. Близость медианы, p95 и p99 говорит о стабильных условиях замера.

Проверяем правила измерения: режимы `train` и `eval`, первый вызов и прогрев, число потоков, размер батча. Функция `ece` считает **ожидаемую ошибку калибровки** (Expected Calibration Error) — средний разрыв между уверенностью модели и долей её верных ответов.

In [ ]:
# 1) train() и eval() дают разные ответы: в режиме обучения dropout случайно выключает нейроны
with torch.no_grad():
    teacher.train()
    out_train = teacher(X_test[:5])
    teacher.eval()
    out_eval = teacher(X_test[:5])
print(f"разница выходов train/eval: {(out_train - out_eval).abs().max():.3f}")

# 2) первый вызов медленнее остальных: память выделяется, кэши заполняются
fresh = copy.deepcopy(teacher)
t = time.perf_counter()
with torch.no_grad():
    fresh(x1)
first = (time.perf_counter() - t) * 1000
print(f"первый вызов: {first:.3f} мс | после прогрева: {latency(fresh, x1)[0]:.3f} мс")

# 3) число потоков: для одного запроса много потоков не нужно, для большого батча — помогает
for threads in [1, 4]:
    torch.set_num_threads(threads)
    print(f"{threads} поток(а): 1 картинка {latency(teacher, x1)[0]:.3f} мс | 256 картинок {latency(teacher, X_test[:256])[0]:.3f} мс")
torch.set_num_threads(4)

# 4) размер батча: задержка растёт медленнее числа картинок, пропускная способность — быстро
for bs in [1, 8, 64, 256]:
    ms = latency(teacher, X_test[:bs])[0]
    print(f"батч {bs:3d}: {ms:.3f} мс на батч | {bs / ms * 1000:9,.0f} картинок в секунду")

def ece(probs, y, bins=10):
    """Expected Calibration Error: средний разрыв между уверенностью и долей верных ответов."""
    conf, pred = probs.max(dim=1)
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)                # примеры с уверенностью в этом интервале
        if m.any():
            gap = (pred[m] == y[m]).float().mean() - conf[m].mean()
            total += m.float().mean() * gap.abs()     # вес интервала — доля его примеров
    return float(total)

with torch.no_grad():
    print(f"ECE учителя: {ece(F.softmax(teacher(X_test), dim=1), y_test):.3f}")

* В режиме `train` выходы отличаются на единицы: забытый `model.eval()` делает модель хуже и случайной.
* Первый вызов медленнее — поэтому прогрев.
* 256 картинок обрабатываются лишь примерно в 10 раз дольше одной: батч делит цену чтения весов на все картинки.
* ECE учителя около 0.016 — уверенность почти точно соответствует доле верных ответов.

## Глава 2. Дистилляция знаний

### 2.2. Дистилляция по логитам

**Дистилляция по логитам**: ученик (один слой из 32 нейронов) учится на смеси кросс-энтропии с меткой и KL с ответами учителя при температуре $T$. Ученику даём всего 100 размеченных картинок и сравниваем обучение только на метках с дистилляцией при $T$ = 1, 4 и 10. Каждый вариант — на трёх наборах.

In [ ]:
with torch.no_grad():
    t_logits = teacher(X_train)                       # ответы учителя на все обучающие картинки

def kd_loss(student_logits, teacher_logits, y, T, alpha):
    """(1 − α)·CE с меткой + α·T²·KL между смягчёнными распределениями учителя и ученика."""
    soft = F.kl_div(F.log_softmax(student_logits / T, dim=1), F.softmax(teacher_logits / T, dim=1),
                    reduction="batchmean") * T * T
    return (1 - alpha) * F.cross_entropy(student_logits, y) + alpha * soft

def student():
    """Ученик: один скрытый слой из 32 нейронов — в 125 раз меньше учителя."""
    return mlp(32, depth=1, p_drop=0.0)

results = {"только метки": [], "T = 1": [], "T = 4": [], "T = 10": []}
for seed in range(3):                                 # три разных набора по 100 картинок
    idx = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(seed))[:100]
    Xs, ys, ts = X_train[idx], y_train[idx], t_logits[idx]
    results["только метки"].append(accuracy(train(student(), X=Xs, y=ys, epochs=500)))
    for T in [1, 4, 10]:
        fn = lambda model, xb, b, T=T: kd_loss(model(xb), ts[b], ys[b], T, alpha=0.9)
        results[f"T = {T}"].append(accuracy(train(student(), X=Xs, y=ys, epochs=500, loss_fn=fn)))
print(f"ученик: {n_params(student()):,} параметров (учитель — {n_params(teacher):,})")
for name, accs in results.items():
    print(f"{name:13s} точность {sum(accs) / 3:.3f}  (запуски: {', '.join(f'{a:.3f}' for a in accs)})")

**Что получилось.** При $T = 1$ дистилляция почти ничего не даёт: ответы учителя почти совпадают с one-hot меткой. При $T$ = 4 и 10 ученик выигрывает около двух пунктов на тех же 100 картинках — он видит, на что похожа каждая цифра.

Учитель «размечает» все 1347 обучающих картинок: ученик учится только на его ответах ($\alpha = 1$). Сравним с тем же учеником на настоящих метках.

In [ ]:
fn = lambda model, xb, b: kd_loss(model(xb), t_logits[b], y_train[b], T=4, alpha=1.0)   # α = 1: только учитель
kd_all = train(student(), X=X_train, loss_fn=fn)
plain = train(student(), X=X_train)                  # тот же ученик на всех настоящих метках
print(f"ученик на 1347 картинках с ответами учителя: {accuracy(kd_all):.3f}")
print(f"ученик на 1347 картинках с настоящими метками: {accuracy(plain):.3f} | учитель: {accuracy(teacher):.3f}")
print(f"размер: ученик {size_kb(kd_all):.0f} КБ, учитель {size_kb(teacher):.0f} КБ | "
      f"задержка на батч 256: ученик {latency(kd_all, X_test[:256])[0]:.3f} мс, учитель {latency(teacher, X_test[:256])[0]:.3f} мс")

**Что получилось.** На одних ответах учителя ученик достигает около 0.956 — без единой настоящей метки. Он весит 12 КБ вместо 1179 и обрабатывает батч примерно в 12 раз быстрее.

## Глава 3. Квантизация

### 3.2. Как устроена квантизация

**Квантизация** вручную: `quantize` — симметричная схема с одним масштабом на тензор или своим масштабом на каждую строку (по каналам). Сравним ошибку на весах второго слоя для 8 и 4 бит и две схемы на активациях после ReLU.

In [ ]:
def quantize(x, bits=8, per_channel=False):
    """Симметричная квантизация: целые от −(2^(b−1)) до 2^(b−1) − 1, ноль остаётся нулём."""
    qmax = 2 ** (bits - 1) - 1                        # для 8 бит — 127
    m = x.abs().amax(dim=1, keepdim=True) if per_channel else x.abs().max()   # свой масштаб у строки или один на всё
    scale = m / qmax
    q = (x / scale).round().clamp(-qmax - 1, qmax)    # целые числа — их и хранят
    return q * scale                                  # деквантизация: обратно в вещественные

W = teacher[3].weight.data                            # веса второго скрытого слоя, 512 × 512
for bits in [8, 4]:
    for pc in [False, True]:
        err = (quantize(W, bits, pc) - W).abs().mean() / W.abs().mean()
        print(f"{bits} бит, {'по каналам ' if pc else 'на тензор  '}: относительная ошибка {err:.4f}")

# активации после ReLU неотрицательные: симметричная схема тратит половину целых на отрицательные
with torch.no_grad():
    a = teacher[:2](X_test)                           # выходы первого ReLU
sym = quantize(a, bits=8)                             # диапазон −127..127, используется только 0..127
scale = a.max() / 255                                 # асимметричная: 0..255, zero point = 0
asym = (a / scale).round().clamp(0, 255) * scale
print(f"активации, 8 бит: симметрично {(sym - a).abs().mean():.5f}, асимметрично {(asym - a).abs().mean():.5f}")

* По каналам ошибка примерно вдвое меньше.
* 8 бит — ошибка 1–2%, 4 бита — уже 18–31%.
* На неотрицательных активациях асимметричная схема вдвое точнее: она использует все 256 значений.

### 3.3. Квантизация после обучения

Сколько бит нужно весам: `quantize_weights` округляет веса всех слоёв, а вычисления остаются вещественными.

In [ ]:
def quantize_weights(model, bits, per_channel=True):
    """Копия модели, у которой веса всех линейных слоёв округлены до bits бит (веса — вещественные, но лежат на сетке)."""
    m = copy.deepcopy(model)
    for layer in m:
        if isinstance(layer, nn.Linear):
            layer.weight.data = quantize(layer.weight.data, bits, per_channel)
    return m

print("бит  на тензор  по каналам")
for bits in [8, 4, 3, 2]:
    print(f"{bits:3d}   {accuracy(quantize_weights(teacher, bits, False)):.3f}      {accuracy(quantize_weights(teacher, bits, True)):.3f}")

**Что получилось.** На 8 и 4 битах точность почти не меняется. На 2 битах один масштаб на тензор разрушает модель (около 0.11 — угадывание): 99.5% весов округляются в ноль. Масштаб по каналам сохраняет около 0.76.

**Калибровка** активаций: масштаб выхода каждого ReLU — по максимуму (min/max) или по 99-му перцентилю. Округляем выходы ReLU до 4 бит **хуком** — функцией, которую PyTorch вызывает после прохода слоя. Затем портим одну из 200 калибровочных картинок: умножаем её яркость на 30.

In [ ]:
def calibrate(X_calib, method, bits):
    """Масштаб для выхода каждого ReLU по калибровочным картинкам: по максимуму или по 99-му перцентилю."""
    scales, h = {}, X_calib
    with torch.no_grad():
        for i, layer in enumerate(teacher):
            h = layer(h)
            if isinstance(layer, nn.ReLU):
                top = h.max() if method == "min/max" else h.flatten().quantile(0.99)
                scales[i] = top / (2 ** bits - 1)     # асимметричная схема: 0..2^b − 1
    return scales

def quantize_activations(scales, bits):
    """Копия учителя, где выход каждого ReLU округляется хуком до bits бит."""
    m = copy.deepcopy(teacher)
    for i, layer in enumerate(m):
        if isinstance(layer, nn.ReLU):
            layer.register_forward_hook(
                lambda mod, inp, out, s=scales[i]: (out / s).round().clamp(0, 2 ** bits - 1) * s)
    return m

X_calib = X_train[:200].clone()                       # калибровочный набор — 200 обучающих картинок
X_outlier = X_calib.clone()
X_outlier[0] *= 30                                    # одна испорченная картинка с огромной яркостью
for name, Xc in [("чистая калибровка", X_calib), ("калибровка с выбросом", X_outlier)]:
    accs = [accuracy(quantize_activations(calibrate(Xc, method, 4), 4)) for method in ["min/max", "перцентиль"]]
    print(f"{name:22s} 4 бита: min/max {accs[0]:.3f} | 99-й перцентиль {accs[1]:.3f}")

**Что получилось.** Одна испорченная картинка из 200 растянула диапазон min/max, и точность упала до угадывания (около 0.1). Перцентиль выброс отбросил — точность прежняя.

### 3.4. Обучение с учётом квантизации

**QAT** (Quantization-Aware Training, обучение с учётом квантизации): `QATLinear` округляет веса в прямом проходе, а градиент пропускает «как будто без округления» (прямой оценщик, STE). Точность сразу после замены — это обычная квантизация после обучения (PTQ); затем 10 эпох QAT.

In [ ]:
class QATLinear(nn.Linear):
    """Линейный слой с «фальшивой» квантизацией весов и прямым пропуском градиента (STE)."""
    bits = 2

    def forward(self, x):
        w = self.weight
        w_q = w + (quantize(w, self.bits, per_channel=True) - w).detach()   # вперёд — квантованные, назад — как будто без округления
        return F.linear(x, w_q, self.bias)

def to_qat(model, bits):
    m = copy.deepcopy(model)
    for i, layer in enumerate(m):
        if isinstance(layer, nn.Linear):
            q = QATLinear(layer.in_features, layer.out_features)
            q.bits = bits
            q.load_state_dict(layer.state_dict())     # стартуем с весов обученного учителя
            m[i] = q
    return m

for bits in [3, 2]:
    qat = to_qat(teacher, bits)
    before = accuracy(qat)                            # это и есть PTQ: просто округлили веса
    qat = train(qat, epochs=10, lr=1e-4)              # дообучение с квантизацией внутри
    print(f"{bits} бита: PTQ {before:.3f} → QAT за 10 эпох {accuracy(qat):.3f}")

**Что получилось.** Для 3 бит QAT не нужен. Для 2 бит 10 эпох QAT возвращают точность исходной модели: веса, которые «знают» об округлении, распределяются по 4 значениям гораздо удачнее.

### 3.5. Форматы и смешанная точность

Квантизация средствами PyTorch: `quantize_dynamic` хранит веса линейных слоёв в INT8 и квантует активации на лету. Сравним размер, точность и скорость FP32, INT8 и BF16.

In [ ]:
engines = torch.backends.quantized.supported_engines
torch.backends.quantized.engine = "qnnpack" if "qnnpack" in engines else engines[-1]   # движок INT8 для этого процессора
int8 = torch.ao.quantization.quantize_dynamic(copy.deepcopy(teacher), {nn.Linear}, dtype=torch.qint8)
bf16 = copy.deepcopy(teacher).to(torch.bfloat16)
print(f"{'модель':10s} {'размер, КБ':>11s} {'точность':>9s} {'1 картинка, мс':>15s} {'256 картинок, мс':>17s}")
for name, model, X in [("FP32", teacher, X_test), ("INT8", int8, X_test), ("BF16", bf16, X_test.bfloat16())]:
    print(f"{name:10s} {size_kb(model):11.0f} {accuracy(model, X):9.3f} {latency(model, X[:1])[0]:15.3f} {latency(model, X[:256])[0]:17.3f}")

**Что получилось.** Размер уменьшился в 4 раза (INT8) и в 2 (BF16), точность та же. А скорость **зависит от процессора**: в лекции INT8 и BF16 оказались медленнее FP32. Сравните со своими числами.

## Глава 4. Прунинг и низкоранговые разложения

### 4.1. Нерегулярный прунинг

**Нерегулярный прунинг**: `prune.global_unstructured` обнуляет нужную долю самых малых по модулю весов по всем слоям, затем 10 эпох дообучения с сохранённой маской.

In [ ]:
import torch.nn.utils.prune as prune

print("доля нулей  без дообучения  после 10 эпох")
for amount in [0.5, 0.8, 0.9, 0.95]:
    pruned = copy.deepcopy(teacher)
    params = [(layer, "weight") for layer in pruned if isinstance(layer, nn.Linear)]
    # глобальный прунинг: один порог для всех слоёв, обнуляется доля amount самых малых по модулю весов
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    before = accuracy(pruned)
    pruned = train(pruned, epochs=10, lr=1e-4)        # маска сохраняется: обнулённые веса остаются нулями
    print(f"{amount:9.0%}  {before:13.3f}  {accuracy(pruned):13.3f}")
zeros = sum((layer.weight == 0).sum().item() for layer in pruned if isinstance(layer, nn.Linear))
print(f"нулей в последней модели: {zeros:,} | задержка на батч 256: {latency(pruned, X_test[:256])[0]:.3f} мс "
      f"(без прунинга {latency(teacher, X_test[:256])[0]:.3f} мс)")

* При 90% нулей точность сразу падает, но дообучение её возвращает.
* При 95% не спасает и дообучение.
* **Скорость не выросла**: нули умножаются, как обычные числа.

### 4.3. Структурный прунинг

**Структурный прунинг**: `shrink` оставляет в каждом скрытом слое `keep` нейронов с наибольшей L1-нормой входящих весов — новая сеть физически меньше. Затем 20 эпох дообучения.

In [ ]:
def shrink(model, keep):
    """Новая сеть, где в каждом скрытом слое оставлены keep нейронов с наибольшей L1-нормой входящих весов."""
    old = [layer for layer in model if isinstance(layer, nn.Linear)]
    new_model = mlp(keep)
    new = [layer for layer in new_model if isinstance(layer, nn.Linear)]
    prev = torch.arange(64)                           # какие входы остались у текущего слоя
    for j, (lo, ln) in enumerate(zip(old, new)):
        if j < len(old) - 1:
            idx = lo.weight.abs().sum(dim=1).topk(keep).indices.sort().values   # самые «важные» нейроны
        else:
            idx = torch.arange(10)                    # у выходного слоя оставляем все 10 классов
        ln.weight.data = lo.weight.data[idx][:, prev].clone()   # строки — оставленные нейроны, столбцы — оставленные входы
        ln.bias.data = lo.bias.data[idx].clone()
        prev = idx
    return new_model.eval()

base_ms = latency(teacher, X_test[:256])[0]
print(f"учитель: 512 нейронов, {n_params(teacher):,} параметров, {base_ms:.3f} мс на батч 256")
for keep in [128, 64, 32]:
    small = shrink(teacher, keep)
    before = accuracy(small)
    small = train(small, epochs=20)
    print(f"{keep:3d} нейрона: {n_params(small):7,} параметров | точность {before:.3f} → {accuracy(small):.3f} после 20 эпох | "
          f"{latency(small, X_test[:256])[0]:.3f} мс")

**Что получилось.** 128 нейронов из 512 после дообучения дают точность учителя при числе параметров в 11 раз меньше — и сеть **реально быстрее**, в отличие от нерегулярного прунинга.

### 4.4. Разреженность N:M и гипотеза лотерейного билета

**Разреженность 2:4**: в каждой четвёрке подряд идущих весов оставляем два наибольших по модулю.

In [ ]:
def two_four(w):
    """В каждой четвёрке подряд идущих весов оставляем два наибольших по модулю."""
    g = w.view(-1, 4)
    top2 = g.abs().topk(2, dim=1).indices
    mask = torch.zeros_like(g).scatter_(1, top2, 1.0)   # 1 — оставить, 0 — обнулить
    return (g * mask).view_as(w)

sparse = copy.deepcopy(teacher)
for layer in sparse:
    if isinstance(layer, nn.Linear) and layer.in_features % 4 == 0:   # у всех наших слоёв 64 или 512 входов
        layer.weight.data = two_four(layer.weight.data)
share = sum((l.weight == 0).float().mean().item() for l in sparse if isinstance(l, nn.Linear)) / 3
print(f"2:4 — нулей в каждом слое: {share:.0%} | точность {accuracy(sparse):.3f} (учитель {accuracy(teacher):.3f})")

**Что получилось.** Половина весов обнулена, а точность упала меньше чем на пункт — даже без дообучения. На видеокартах NVIDIA с поддержкой 2:4 это дало бы ускорение в 1.5–2 раза.

### 4.5. Низкоранговые разложения

**Низкоранговое разложение**: сингулярное разложение (SVD) второго слоя 512 × 512, ранги для разной доли «энергии» и замена слоя двумя узкими с рангом 16, 64 и 137 — без дообучения.

In [ ]:
W = teacher[3].weight.data                            # слой 512 × 512: 262 144 веса
U, S, Vh = torch.linalg.svd(W)                        # W = U · diag(S) · Vh
energy = (S ** 2).cumsum(0) / (S ** 2).sum()          # доля «энергии» в первых k сингулярных числах
for p in [0.5, 0.8, 0.9, 0.95]:
    print(f"{p:.0%} энергии — ранг {int((energy < p).sum()) + 1}")

for r in [16, 64, 137]:
    lr_model = copy.deepcopy(teacher)
    A = nn.Linear(512, r, bias=False)                 # сначала сжатие до r чисел
    B = nn.Linear(r, 512)                             # потом расширение обратно
    A.weight.data = torch.diag(S[:r]) @ Vh[:r]        # r × 512
    B.weight.data = U[:, :r].clone()                  # 512 × r
    B.bias.data = teacher[3].bias.data.clone()
    lr_model[3] = nn.Sequential(A, B)                 # B(A(x)) ≈ W·x + b
    print(f"ранг {r:3d}: весов в слое {512 * r * 2:7,} вместо 262,144 | точность {accuracy(lr_model):.3f}")

**Что получилось.** Критерий энергии требует ранга 137 для 90%, но даже ранг 16 — в 16 раз меньше весов — сохраняет точность без дообучения: учитель сильно избыточен для цифр.

## Глава 5. Оптимизация графа и экспорт

### 5.1. Слияние и свёртка операций

**Свёртка BatchNorm** в предыдущий линейный слой: новые веса $W' = \frac{\gamma}{\sigma} W$ и сдвиг $b' = \frac{\gamma}{\sigma}(b - \mu) + \beta$. Сравним выходы и время.

In [ ]:
torch.manual_seed(0)
lin, bn = nn.Linear(64, 512), nn.BatchNorm1d(512)
bn.running_mean.uniform_(-1, 1)                       # как будто сеть уже обучена: накопленные статистики
bn.running_var.uniform_(0.5, 2)
bn.weight.data.uniform_(0.5, 1.5)                     # γ
bn.bias.data.uniform_(-0.5, 0.5)                      # β
two_layers = nn.Sequential(lin, bn).eval()

k = bn.weight / torch.sqrt(bn.running_var + bn.eps)   # γ / σ для каждого нейрона
folded = nn.Linear(64, 512)
folded.weight.data = lin.weight.data * k[:, None]     # W' = (γ/σ)·W
folded.bias.data = (lin.bias.data - bn.running_mean) * k + bn.bias.data   # b' = (γ/σ)(b − μ) + β
with torch.no_grad():
    print(f"наибольшая разница выходов: {(two_layers(X_test) - folded(X_test)).abs().max():.1e}")
print(f"батч 256: Linear + BatchNorm {latency(two_layers, X_test[:256])[0]:.3f} мс | после свёртки {latency(folded, X_test[:256])[0]:.3f} мс")

**Что получилось.** Выходы совпали до ошибок округления, а одна операция вместо двух работает быстрее.

### 5.2. TorchScript

**TorchScript**: модель `Branchy` с ветвлением, зависящим от входа. Трассируем её на картинках с положительной суммой и проверяем на входе с отрицательной; затем оптимизируем учителя для инференса.

In [ ]:
class Branchy(nn.Module):
    """Модель с ветвлением, зависящим от данных."""
    def __init__(self):
        super().__init__()
        self.net = mlp(64)

    def forward(self, x):
        if x.sum() > 0:                               # условие зависит от входа
            return self.net(x)
        return -self.net(x)

branchy = Branchy().eval()
traced = torch.jit.trace(branchy, X_test[:4])         # записываем операции на примере с положительной суммой
scripted = torch.jit.script(branchy)                  # компилируем код целиком, вместе с if
negative = -X_test[:4]                                # вход, где должна сработать вторая ветка
with torch.no_grad():
    print(f"трассировка верна на новом входе: {torch.allclose(traced(negative), branchy(negative))} | "
          f"скриптинг: {torch.allclose(scripted(negative), branchy(negative))}")

optimized = torch.jit.optimize_for_inference(torch.jit.script(teacher))   # заморозка весов и оптимизации графа
with torch.no_grad():
    print(f"выходы совпадают: {torch.allclose(optimized(X_test), teacher(X_test), atol=1e-5)}")
print(f"1 картинка: обычный PyTorch {latency(teacher, x1)[0]:.3f} мс | TorchScript {latency(optimized, x1)[0]:.3f} мс")

**Что получилось.** Трассировка «запомнила» только одну ветку и **молча** ошиблась на другом входе; скриптинг сохранил условие. Оптимизированный учитель на одной картинке быстрее: нет накладных расходов Python.

### 5.3. ONNX и ONNX Runtime

**ONNX и ONNX Runtime**: экспорт учителя с динамическим размером батча, сессия ONNX Runtime, сравнение выходов с PyTorch и время на 1 и 256 картинках. Файл `teacher.onnx` появится рядом с ноутбуком.

In [ ]:
import numpy as np
import onnxruntime as ort

torch.onnx.export(teacher, x1, "teacher.onnx",
                  input_names=["pixels"], output_names=["logits"],
                  dynamic_axes={"pixels": {0: "batch"}, "logits": {0: "batch"}},   # размер батча — любой
                  opset_version=17, dynamo=False)
print(f"файл teacher.onnx: {os.path.getsize('teacher.onnx') / 1024:.0f} КБ")

options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL   # все оптимизации графа
options.intra_op_num_threads = 4
session = ort.InferenceSession("teacher.onnx", options, providers=["CPUExecutionProvider"])
run = lambda x: session.run(None, {"pixels": x.numpy()})[0]   # вход и выход — массивы NumPy

with torch.no_grad():
    ref = teacher(X_test).numpy()
out = run(X_test)
print(f"наибольшая разница с PyTorch: {np.abs(out - ref).max():.1e} | совпадение ответов: {(out.argmax(1) == ref.argmax(1)).mean():.3f}")
np.testing.assert_allclose(out, ref, atol=1e-3, rtol=1e-2)   # упадёт, если разница больше допустимой
for name, fn in [("PyTorch", teacher), ("ONNX Runtime", run)]:
    print(f"{name:13s} 1 картинка {latency(fn, x1)[0]:.3f} мс | 256 картинок {latency(fn, X_test[:256])[0]:.3f} мс")

**Что получилось.** Выходы совпали до $10^{-4}$, ответы — на всех картинках. Какой движок быстрее на одной картинке и на батче, зависит от процессора: в лекции ONNX Runtime выиграл на одном запросе и проиграл на батче из 256.

## Глава 6. Test-Time Augmentation

### 6.2. Проверка

**Test-Time Augmentation**: `tta` усредняет вероятности по исходной картинке и четырём сдвигам на пиксель. Сравним учителя (сдвигов при обучении не видел) и такую же сеть, обученную со случайными сдвигами, на чистом тесте и на тесте с сильным шумом. В конце — **MC-Dropout**: 20 прогонов с включённым dropout.

In [ ]:
def shift(x, dx, dy):
    """Сдвиг картинок на dx, dy пикселей; освободившиеся края заполняются нулями."""
    img = F.pad(x.view(-1, 1, 8, 8), (1, 1, 1, 1))
    return img[:, 0, 1 + dy:9 + dy, 1 + dx:9 + dx].reshape(-1, 64)

def tta(model, X, shifts=((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1))):
    """Среднее вероятностей по исходной картинке и четырём сдвигам."""
    with torch.no_grad():
        return torch.stack([F.softmax(model(shift(X, dx, dy)), dim=1) for dx, dy in shifts]).mean(dim=0)

def shift_loss(model, xb, b):
    d = torch.randint(-1, 2, (2,))                    # случайный сдвиг батча на −1..1 пиксель
    return F.cross_entropy(model(shift(xb, int(d[0]), int(d[1]))), y_train[b])

torch.manual_seed(0)
aug_model = train(mlp(512), loss_fn=shift_loss)       # та же сеть, обученная со сдвигами
torch.manual_seed(0)
X_noisy = (X_test + 0.3 * torch.randn_like(X_test)).clamp(0, 1)   # тест с сильным шумом

print(f"{'модель':22s} {'данные':8s} {'без TTA':>8s} {'с TTA':>6s}   ECE без/с TTA")
for mname, model in [("учитель (без сдвигов)", teacher), ("обучена со сдвигами", aug_model)]:
    for dname, X in [("чистые", X_test), ("с шумом", X_noisy)]:
        with torch.no_grad():
            p = F.softmax(model(X), dim=1)
        pt = tta(model, X)
        acc = lambda probs: (probs.argmax(1) == y_test).float().mean().item()
        print(f"{mname:22s} {dname:8s} {acc(p):8.3f} {acc(pt):6.3f}   {ece(p, y_test):.3f} / {ece(pt, y_test):.3f}")

# MC-Dropout: dropout включён при инференсе, 20 прогонов одного и того же входа
aug_model.train()
with torch.no_grad():
    runs = torch.stack([F.softmax(aug_model(X_noisy), dim=1) for _ in range(20)])   # 20 × 450 × 10
aug_model.eval()
mean = runs.mean(dim=0)
pred = mean.argmax(dim=1)
spread = runs.std(dim=0).gather(1, pred[:, None]).squeeze(1)   # разброс вероятности выбранного класса
wrong = pred != y_test
print(f"MC-Dropout на шумных: точность {(~wrong).float().mean():.3f} | разброс у верных {spread[~wrong].mean():.3f}, у ошибок {spread[wrong].mean():.3f}")

* Учитель с TTA **хуже**: сдвинутые картинки для него незнакомые.
* Модель, обученная со сдвигами, на шумных данных от TTA выигрывает около 9 пунктов, и калибровка улучшается.
* MC-Dropout: у ошибок разброс ответов заметно больше, чем у верных, — по нему можно находить ненадёжные ответы.